# Laboratorio AWS — Predicción de temperatura de un transformador
## Notebook 06 — Introducción a Pipelines y MLOps

### Objetivo

Construir y validar la lógica de un flujo MLOps reproducible.

En este notebook no ejecutaremos Training Jobs.

La cuenta puede tener restricciones de cómputo, pero podemos demostrar el principio de orquestación:

```text
Datos
  ↓
Preparación
  ↓
Entrenamiento
  ↓
Evaluación
  ↓
Registro
  ↓
Aprobación
  ↓
Despliegue
```

### Idea central

Una pipeline no es el modelo.

Es el **flujo que gobierna las etapas del ciclo de vida del modelo**.

## Paso 1. Arquitectura MLOps

El flujo que hemos construido hasta ahora es:

```text
Notebook 01
Generación y exploración
        ↓
Notebook 02
S3 + entrenamiento local
        ↓
Notebook 03
Training Job administrado
        ↓
Notebook 04
Inferencia
        ↓
Notebook 05
Model Registry
        ↓
Notebook 06
Pipeline / MLOps
```

Si una capacidad administrada no está disponible, el artefacto producido anteriormente permite continuar.

In [ ]:
import json
from pathlib import Path

import boto3
import pandas as pd

BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

print(f"Región: {REGION}")
print(f"Bucket: {BUCKET}")

## Paso 2. Definir las etapas de la pipeline

Primero construimos la lógica como un DAG conceptual.

No se crean recursos AWS.

In [ ]:
pipeline_steps = [
    {
        "name": "ValidarDatos",
        "tipo": "validacion",
        "entrada": "s3://siemens-ml-lab-aiarch-2026/data/raw/transformadores.csv"
    },
    {
        "name": "PrepararTrainTest",
        "tipo": "preparacion",
        "entrada": "data/raw",
        "salida": ["data/train", "data/test"]
    },
    {
        "name": "EntrenarModelo",
        "tipo": "entrenamiento",
        "entrada": ["data/train", "data/test"],
        "salida": "model/"
    },
    {
        "name": "EvaluarModelo",
        "tipo": "evaluacion",
        "entrada": "model/",
        "salida": "output/metricas_modelos.csv"
    },
    {
        "name": "RegistrarModelo",
        "tipo": "gobierno",
        "entrada": "model/",
        "salida": "Model Registry"
    }
]

print(json.dumps(pipeline_steps, indent=2, ensure_ascii=False))

## Paso 3. Representar las dependencias

Una pipeline se comporta como un grafo dirigido.

Las dependencias establecen el orden lógico de ejecución.

In [ ]:
dependencies = {
    "ValidarDatos": [],
    "PrepararTrainTest": ["ValidarDatos"],
    "EntrenarModelo": ["PrepararTrainTest"],
    "EvaluarModelo": ["EntrenarModelo"],
    "RegistrarModelo": ["EvaluarModelo"]
}

for step, parents in dependencies.items():
    print(f"{step} <- {parents if parents else 'inicio'}")

## Paso 4. Validar los artefactos reales

La pipeline conceptual debe corresponder con los artefactos que ya existen.

Verificaremos S3 sin crear cómputo.

In [ ]:
prefixes = [
    "data/raw/",
    "data/train/",
    "data/test/",
    "model/",
    "output/"
]

for prefix in prefixes:
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    keys = [x["Key"] for x in response.get("Contents", [])]

    print(f"{prefix}")
    for key in keys:
        print(" -", key)

## Paso 5. Construir una definición reproducible

Guardamos la definición del flujo como JSON.

Esto no ejecuta la pipeline.

La definición sirve para documentar qué debe ocurrir y en qué orden.

In [ ]:
pipeline_definition = {
    "pipeline_name": "transformadores-ml-lab",
    "version": 1,
    "region": REGION,
    "steps": pipeline_steps,
    "dependencies": dependencies,
    "model_group": "transformadores-temperatura"
}

pipeline_file = Path("pipeline_definition.json")
pipeline_file.write_text(
    json.dumps(pipeline_definition, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Definición guardada en: {pipeline_file.resolve()}")

## Paso 6. Concepto de ejecución administrada

En una cuenta con las cuotas y permisos necesarios, esta definición conceptual se puede convertir en una SageMaker Pipeline real.

Una ejecución administrada puede crear recursos como:

- Processing Jobs;
- Training Jobs;
- evaluación;
- registro del modelo;
- condiciones de aprobación;
- posteriores pasos de despliegue.

Para este laboratorio **no ejecutaremos esos recursos**.

El objetivo es comprender la arquitectura y dejar la lógica lista para una cuenta con capacidad de cómputo.

In [ ]:
print("Pipeline preparada conceptualmente.")
print()
print("Flujo:")
print("Datos → Preparación → Entrenamiento → Evaluación → Registro → Despliegue")
print()
print("Modo de ejecución actual: validación local y documentación.")
print("No se crea Training Job ni Endpoint.")

## Paso 7. Cierre del Notebook 06 — visión MLOps

Hemos pasado de un modelo aislado a un ciclo de vida:

```text
                 ┌───────────────┐
                 │     Datos     │
                 └───────┬───────┘
                         ↓
                 ┌───────────────┐
                 │  Preparación  │
                 └───────┬───────┘
                         ↓
                 ┌───────────────┐
                 │ Entrenamiento │
                 └───────┬───────┘
                         ↓
                 ┌───────────────┐
                 │   Evaluación  │
                 └───────┬───────┘
                         ↓
                 ┌───────────────┐
                 │Model Registry │
                 └───────┬───────┘
                         ↓
                 ┌───────────────┐
                 │   Despliegue  │
                 └───────────────┘
```

### Resultado

Los cuatro notebooks 03–06 permiten completar **28 pasos**, siete por notebook, sin depender de una cuota de entrenamiento administrado.

La limitación actual de AWS se convierte en parte del aprendizaje:

**Cloud no significa que todos los recursos estén siempre disponibles; significa diseñar el flujo para separar datos, modelos, cómputo, gobierno y despliegue.**